In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('cleaned_recipes.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157973 entries, 0 to 157972
Data columns (total 22 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   RecipeId                    157973 non-null  int64  
 1   Name                        157973 non-null  object 
 2   CookTime                    157973 non-null  object 
 3   PrepTime                    157973 non-null  object 
 4   TotalTime                   157973 non-null  object 
 5   Description                 157973 non-null  object 
 6   Images                      157973 non-null  object 
 7   RecipeCategory              157973 non-null  object 
 8   Keywords                    157973 non-null  object 
 9   RecipeIngredientQuantities  157973 non-null  object 
 10  RecipeIngredientParts       157973 non-null  object 
 11  Calories                    157973 non-null  float64
 12  FatContent                  157973 non-null  float64
 13  SaturatedFatCo

In [4]:
df.head()

,RecipeId,Name,CookTime,PrepTime,TotalTime,Description,Images,RecipeCategory,Keywords,RecipeIngredientQuantities,...,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeInstructions
0,1,Low-Fat Berry Blue Frozen Dessert,PT24H,PT45M,PT24H45M,Make and share this Low-Fat Berry Blue Frozen ...,"c(""https://img.sndimg.com/food/image/upload/w_...",Frozen Desserts,"c(""Dessert"", ""Low Protein"", ""Low Cholesterol"",...","c(""4"", ""1/4"", ""1"", ""1"")",...,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,2,Biryani,PT25M,PT4H,PT4H25M,Make and share this Biryani recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_...",Chicken Breast,"c(""Chicken Thigh & Leg"", ""Chicken"", ""Poultry"",...","c(""1"", ""4"", ""2"", ""2"", ""8"", ""1/4"", ""8"", ""1/2"", ...",...,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,"c(""Soak saffron in warm milk for 5 minutes and..."
2,3,Best Lemonade,PT5M,PT30M,PT35M,This is from one of my first Good House Keepi...,"c(""https://img.sndimg.com/food/image/upload/w_...",Beverages,"c(""Low Protein"", ""Low Cholesterol"", ""Healthy"",...","c(""1 1/2"", ""1"", NA, ""1 1/2"", NA, ""3/4"")",...,0.2,0.0,0.0,1.8,81.5,0.4,77.2,0.3,4.0,"c(""Into a 1 quart Jar with tight fitting lid, ..."
3,4,Carina's Tofu-Vegetable Kebabs,PT20M,PT24H,PT24H20M,This dish is best prepared a day in advance to...,"c(""https://img.sndimg.com/food/image/upload/w_...",Soy/Tofu,"c(""Beans"", ""Vegetable"", ""Low Cholesterol"", ""We...","c(""12"", ""1"", ""2"", ""1"", ""10"", ""1"", ""3"", ""2"", ""2...",...,24.0,3.8,0.0,1558.6,64.2,17.3,32.1,29.3,2.0,"c(""Drain the tofu, carefully squeezing out exc..."
4,5,Cabbage Soup,PT30M,PT20M,PT50M,Make and share this Cabbage Soup recipe from F...,"c(""https://img.sndimg.com/food/image/upload/w_...",Vegetable,"c(""Low Protein"", ""Vegan"", ""Low Cholesterol"", ""...","c(""46"", ""4"", ""1"", ""2"", ""1"")",...,0.4,0.1,0.0,959.3,25.1,4.8,17.7,4.3,4.0,"c(""Mix everything together and bring to a boil..."


In [ ]:
import importlib
import subprocess
import sys

required_pkgs = [
    "torch",
    "transformers",
    "accelerate",
    "sentencepiece",
    "isodate",
    "scikit-learn",
]

missing = []
for pkg in required_pkgs:
    module_name = "sklearn" if pkg == "scikit-learn" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import ast
import csv
import re
from io import StringIO
from typing import List

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainingArguments, T5ForConditionalGeneration
import isodate

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

Installing missing packages: ['torch', 'transformers', 'accelerate', 'sentencepiece', 'isodate']


In [7]:
NUTRITION_COLS = [
    "Calories",
    "FatContent",
    "SaturatedFatContent",
    "CholesterolContent",
    "SodiumContent",
    "CarbohydrateContent",
    "FiberContent",
    "SugarContent",
    "ProteinContent",
]

UNIT_WORDS = {
    "tsp", "teaspoon", "teaspoons", "tbsp", "tablespoon", "tablespoons",
    "cup", "cups", "oz", "ounce", "ounces", "lb", "lbs", "pound", "pounds",
    "g", "gram", "grams", "kg", "ml", "l", "liter", "liters", "pinch", "dash",
    "clove", "cloves", "slice", "slices", "can", "cans", "package", "packages"
}

def parse_list_like(value) -> List[str]:
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    if text.startswith("c(") and text.endswith(")"):
        inner = text[2:-1].strip()
        reader = csv.reader(StringIO(inner), delimiter=",", quotechar='"', skipinitialspace=True)
        row = next(reader, [])
        out = []
        for token in row:
            token = token.strip().replace('""', '"')
            if token in {"", "NA", "None", "nan"}:
                continue
            out.append(token)
        return out

    return [text]


def parse_iso_minutes(value) -> float:
    if pd.isna(value):
        return 0.0
    text = str(value).strip()
    if not text:
        return 0.0
    try:
        duration = isodate.parse_duration(text)
        if hasattr(duration, "total_seconds"):
            return float(duration.total_seconds() / 60.0)
    except Exception:
        pass

    m = re.match(r"P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?)?", text)
    if not m:
        return 0.0
    days = int(m.group(1) or 0)
    hours = int(m.group(2) or 0)
    minutes = int(m.group(3) or 0)
    return float(days * 24 * 60 + hours * 60 + minutes)


def normalize_ingredient(token: str) -> str:
    s = token.lower().strip()
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"^\s*(?:\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)\s*", "", s)
    s = re.sub(r"^\s*(?:x|about|approx\.?|approximately)\s+", "", s)

    parts = s.split()
    while parts and re.fullmatch(r"\d+(?:\.\d+)?", parts[0]):
        parts.pop(0)
    while parts and parts[0] in UNIT_WORDS:
        parts.pop(0)

    s = " ".join(parts)
    s = re.sub(r"[^a-zA-Z\s-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip(" -")
    return s


def build_input_prompt(ingredients: List[str], total_minutes: float) -> str:
    cleaned = [normalize_ingredient(x) for x in ingredients]
    cleaned = [x for x in cleaned if x]
    cleaned = list(dict.fromkeys(cleaned))
    if len(cleaned) > 40:
        cleaned = cleaned[:40]
    prompt = f"ingredients: {', '.join(cleaned)}"
    if total_minutes > 0:
        prompt += f" | ready_in_min: {int(total_minutes)}"
    return prompt


def build_target_text(name: str, instructions: List[str]) -> str:
    steps = [s.strip() for s in instructions if s and s.strip()]
    if len(steps) > 25:
        steps = steps[:25]
    body = "\n".join(f"{i+1}. {step}" for i, step in enumerate(steps))
    title = str(name).strip() if pd.notna(name) else "Unknown Recipe"
    return f"name: {title}\nsteps:\n{body}"

df_work = df.copy()
df_work["_ingredient_list"] = df_work["RecipeIngredientParts"].apply(parse_list_like)
df_work["_instruction_list"] = df_work["RecipeInstructions"].apply(parse_list_like)
df_work["_total_minutes"] = df_work["TotalTime"].apply(parse_iso_minutes)

df_work["RecipeServings"] = pd.to_numeric(df_work["RecipeServings"], errors="coerce")
serving_median = df_work["RecipeServings"].dropna().median()
if pd.isna(serving_median) or serving_median <= 0:
    serving_median = 4.0
df_work["RecipeServings"] = df_work["RecipeServings"].fillna(serving_median)
df_work["RecipeServings"] = df_work["RecipeServings"].clip(lower=1.0)

for col in NUTRITION_COLS:
    df_work[col] = pd.to_numeric(df_work[col], errors="coerce")
    df_work[col] = df_work[col] / df_work["RecipeServings"]

df_work["input_text"] = df_work.apply(
    lambda r: build_input_prompt(r["_ingredient_list"], r["_total_minutes"]),
    axis=1,
)
df_work["target_text"] = df_work.apply(
    lambda r: build_target_text(r["Name"], r["_instruction_list"]),
    axis=1,
)

df_work = df_work[df_work["input_text"].str.len() > 15]
df_work = df_work[df_work["target_text"].str.len() > 20]
df_work = df_work.dropna(subset=NUTRITION_COLS).reset_index(drop=True)

print("Prepared rows:", len(df_work))
print(df_work[["input_text", "target_text"]].head(2))

Prepared rows: 157973
                                          input_text  \
0  ingredients: blueberries, granulated sugar, va...   
1  ingredients: saffron, milk, hot green chili pe...   

                                         target_text  
0  name: Low-Fat Berry Blue Frozen Dessert\nsteps...  
1  name: Biryani\nsteps:\n1. Soak saffron in warm...  


In [8]:
MAX_ROWS = 30000  # reduce for faster experimentation; raise for full training
if len(df_work) > MAX_ROWS:
    df_model = df_work.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
else:
    df_model = df_work.copy()

idx = np.arange(len(df_model))
train_idx, val_idx = train_test_split(idx, test_size=0.1, random_state=SEED)

train_df = df_model.iloc[train_idx].reset_index(drop=True)
val_df = df_model.iloc[val_idx].reset_index(drop=True)

scaler = StandardScaler()
train_targets = scaler.fit_transform(train_df[NUTRITION_COLS].to_numpy(dtype=np.float32))
val_targets = scaler.transform(val_df[NUTRITION_COLS].to_numpy(dtype=np.float32))

MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class RecipeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, y_scaled: np.ndarray):
        self.frame = frame
        self.y = y_scaled

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        model_in = tokenizer(
            row["input_text"],
            truncation=True,
            max_length=MAX_INPUT_LEN,
        )
        model_out = tokenizer(
            text_target=row["target_text"],
            truncation=True,
            max_length=MAX_TARGET_LEN,
        )

        return {
            "input_ids": model_in["input_ids"],
            "attention_mask": model_in["attention_mask"],
            "labels": model_out["input_ids"],
            "reg_targets": torch.tensor(self.y[idx], dtype=torch.float32),
        }

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=MODEL_NAME, padding=True)

def multitask_collate(features):
    reg_targets = torch.stack([f["reg_targets"] for f in features])
    core = [{k: v for k, v in f.items() if k != "reg_targets"} for f in features]
    batch = data_collator(core)
    batch["labels"][batch["labels"] == tokenizer.pad_token_id] = -100
    batch["reg_targets"] = reg_targets
    return batch

train_ds = RecipeDataset(train_df, train_targets)
val_ds = RecipeDataset(val_df, val_targets)

print(f"Model rows: {len(df_model)} | Train size: {len(train_ds)} | Val size: {len(val_ds)}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Model rows: 30000 | Train size: 27000 | Val size: 3000


In [9]:
class MultiTaskFlanT5(nn.Module):
    def __init__(self, model_name: str, num_targets: int, reg_weight: float = 0.5):
        super().__init__()
        self.backbone = T5ForConditionalGeneration.from_pretrained(model_name)
        hidden = self.backbone.config.d_model
        self.reg_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, num_targets),
        )
        self.reg_weight = reg_weight

    def forward(self, input_ids=None, attention_mask=None, labels=None, reg_targets=None, **kwargs):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True,
        )

        enc_hidden = out.encoder_last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (enc_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        reg_pred = self.reg_head(pooled)

        reg_loss = None
        loss = out.loss
        if reg_targets is not None:
            reg_loss = nn.functional.mse_loss(reg_pred, reg_targets)
            loss = loss + self.reg_weight * reg_loss

        return {
            "loss": loss,
            "logits": out.logits,
            "reg_logits": reg_pred,
            "ce_loss": out.loss.detach() if out.loss is not None else None,
            "reg_loss": reg_loss.detach() if reg_loss is not None else None,
        }

    @torch.no_grad()
    def generate_text(self, input_ids, attention_mask, **kwargs):
        return self.backbone.generate(input_ids=input_ids, attention_mask=attention_mask, **kwargs)

model = MultiTaskFlanT5(MODEL_NAME, num_targets=len(NUTRITION_COLS), reg_weight=0.5)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./recipe_multitask_ckpt",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    max_steps=120,  # quick run; increase for stronger quality
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=60,
    save_strategy="no",  # avoids shared-tensor save errors for custom wrapper modules
    logging_steps=20,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=multitask_collate,
)

train_result = trainer.train()
print(train_result.metrics)

Step,Training Loss,Validation Loss
60,2.808837,2.826021
120,2.928802,2.797617


RuntimeError: on_train_begin must be called before on_evaluate

In [13]:
inv_scale = lambda y_scaled: scaler.inverse_transform(y_scaled)

sample = val_df.iloc[0]
enc = tokenizer(sample["input_text"], return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
enc = {k: v.to(device) for k, v in enc.items()}

model.eval()
with torch.no_grad():
    gen_ids = model.generate_text(
        enc["input_ids"],
        enc["attention_mask"],
        max_length=MAX_TARGET_LEN,
        num_beams=4,
    )
    enc_hidden = model.backbone.encoder(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        return_dict=True,
    ).last_hidden_state
    pooled = (enc_hidden * enc["attention_mask"].unsqueeze(-1)).sum(dim=1) / enc["attention_mask"].sum(dim=1, keepdim=True).clamp(min=1)
    pred_scaled = model.reg_head(pooled).cpu().numpy()

pred_recipe = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
pred_nutrition = inv_scale(pred_scaled)[0]

print("Input:")
print(sample["input_text"])
print("\nGenerated recipe:")
print(pred_recipe)
print("\nPredicted nutrition (per serving):")
for c, v in zip(NUTRITION_COLS, pred_nutrition):
    print(f"{c}: {v:.2f}")

print("\nGround truth nutrition (per serving):")
print(sample[NUTRITION_COLS].to_string())

Input:
ingredients: sun-dried tomatoes, fresh chili pepper, lemongrass, lime, mozzarella cheese, rocket, mayonnaise, salt, pepper | ready_in_min: 28

Generated recipe:
name: Grilled Sun-Dried Tomatoes steps: 1. Preheat oven to 350°F. 2. In a large bowl, combine the tomatoes, chili pepper, lemongrass, lime, and pepper. 3. In a large bowl, combine the tomatoes, chili pepper, lemongrass, lime, and mayonnaise. 4. In a large bowl, combine the tomatoes, chili pepper, lemongrass, lime, and mayonnaise. 5. In a large bowl, combine the tomatoes, chili pepper, lemongrass, lime, and mayonnaise. 6. In a large bowl, combine the tomatoes, chili pepper, lemongrass, lime, and mayonnaise. 7. In a large bowl, combine the tomatoes, chili pepper, lemongrass, lime, and mayonnaise. 8. In a large bowl, combine the tomatoes, chili pepper, lemongrass, lime, and mayonnaise. 9. In a large bowl, combine the tomatoes, chili pepper, lemongrass, lime, and mayonnaise. 10. In a large bowl, combine the tomatoes, chili p